<a href="https://colab.research.google.com/github/DarioCorona/personal/blob/root/Evaluacion1.1_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación integradora final

## SQL aplicado al análisis de mantenimiento

**Entorno:** Google Colab  
**Motor SQL:** DuckDB  
**Tabla:** `ordenes_trabajo`

En esta evaluación construirás una consulta SQL que genere indicadores de mantenimiento agrupados por tipo de equipo. El notebook permite subir el CSV desde tu computadora, prepara los datos reales, crea una tabla temporal en DuckDB y proporciona un calificador automático alineado con la rúbrica.

> **Integridad académica:** escribe una sola solución SQL propia. No modifiques el dataframe fuente, la función de evaluación ni los resultados después de ejecutar la consulta.

## Instrucciones antes de comenzar

1. Abre este archivo en **Google Colab**.
2. Ejecuta las celdas en orden con `Shift + Enter`.
3. Cuando se solicite, sube el CSV original desde tu computadora.
4. Escribe la solución exclusivamente en la celda `CONSULTA_ESTUDIANTE`.
5. Ejecuta el calificador y conserva sus resultados en el notebook.
6. Descarga el `.ipynb` resuelto antes de cerrar la sesión.

El archivo fuente contiene 39 columnas y 4,427 órdenes de trabajo válidas. La exportación también contiene filas vacías generadas por Excel; la preparación las elimina usando el número de OT como criterio.

> **Importante:** DuckDB trabaja en memoria y los archivos subidos a Colab son temporales. Si Colab reinicia la sesión, vuelve a ejecutar el notebook desde el principio y sube nuevamente el CSV.

## 1. Preparación del entorno

Google Colab ya incluye Python y pandas. La siguiente celda instala DuckDB en la sesión actual.

In [1]:
%pip install -q duckdb

In [2]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda valor: f"{valor:,.4f}")

print(f"DuckDB: {duckdb.__version__}")
print("Entorno de Google Colab preparado correctamente.")

DuckDB: 1.3.2
Entorno de Google Colab preparado correctamente.


## 2. Subida y preparación del archivo real

Ejecuta la siguiente celda y selecciona el CSV de órdenes de trabajo desde tu computadora.

Para esta evaluación se conservan solamente las columnas necesarias. Los nombres originales se normalizan al esquema SQL utilizado en el enunciado:

| Columna original | Columna SQL | Descripción |
|---|---|---|
| `Nº OT` | `num_ot` | Identificador único de la orden |
| `Objeto` | `objeto` | Código del activo o equipo |
| `Descripción Tipo Objeto` | `desc_tipo_objeto` | Agrupación solicitada |
| `Tipo Trabajo` | `tipo_trabajo` | Código del trabajo, como `MC` o `MP` |
| `Total Horas Hombre` | `total_hh` | Horas hombre registradas |

In [6]:
# Abre el selector de archivos de Google Colab.
archivos_subidos = files.upload()

archivos_csv = [
    nombre
    for nombre in archivos_subidos
    if nombre.lower().endswith(".csv")
]

if not archivos_csv:
    if archivos_subidos:
        uploaded_filename = list(archivos_subidos.keys())[0]
        if uploaded_filename.lower().endswith(".xlsx"):
            raise FileNotFoundError(f"Se esperaba un archivo CSV, pero se subió '{uploaded_filename}'. Por favor, sube el archivo CSV original.")
        else:
            raise FileNotFoundError(f"No se subió ningún archivo CSV. El archivo subido fue '{uploaded_filename}'. Por favor, sube el archivo CSV original.")
    else:
        raise FileNotFoundError("No se subió ningún archivo CSV. Por favor, asegúrate de seleccionar un archivo.")

ARCHIVO_CSV = archivos_csv[0]

columnas_origen = [
    "Nº OT",
    "Objeto",
    "Descripción Tipo Objeto",
    "Tipo Trabajo",
    "Total Horas Hombre",
]

renombrado = {
    "Nº OT": "num_ot",
    "Objeto": "objeto",
    "Descripción Tipo Objeto": "desc_tipo_objeto",
    "Tipo Trabajo": "tipo_trabajo",
    "Total Horas Hombre": "total_hh",
}

df_ot = pd.read_csv(
    ARCHIVO_CSV,
    encoding="utf-8-sig",
    usecols=columnas_origen,
    low_memory=False,
).rename(columns=renombrado)

# Una fila representa una OT solamente cuando tiene número de orden.
df_ot = df_ot.loc[df_ot["num_ot"].notna()].copy()

# Conversión explícita de los tipos utilizados en SQL.
df_ot["num_ot"] = df_ot["num_ot"].astype("int64")
df_ot["objeto"] = df_ot["objeto"].astype("string")
df_ot["desc_tipo_objeto"] = df_ot["desc_tipo_objeto"].astype("string")
df_ot["tipo_trabajo"] = df_ot["tipo_trabajo"].astype("string")
df_ot["total_hh"] = pd.to_numeric(df_ot["total_hh"], errors="coerce")

print(f"Archivo cargado: {ARCHIVO_CSV}")
print(f"Órdenes válidas: {len(df_ot):,}")
display(df_ot.head())

Saving ordenes de trabajo historicas del año 2024.csv to ordenes de trabajo historicas del año 2024 (1).csv
Archivo cargado: ordenes de trabajo historicas del año 2024 (1).csv
Órdenes válidas: 4,427


,num_ot,objeto,desc_tipo_objeto,total_hh,tipo_trabajo
0,90748,51665,Mecanico,7.0000,MP
1,90749,51666,Mecanico,7.0000,MP
2,90744,56724,Bomba,36.0000,MP
3,94444,43610,Bomba,8.0000,MP
4,94445,43612,Bomba,4.0000,MP


In [7]:
diagnostico = pd.DataFrame({
    "tipo_python": df_ot.dtypes.astype(str),
    "nulos": df_ot.isna().sum(),
    "valores_unicos": df_ot.nunique(dropna=True),
})

display(diagnostico)
print("Distribución de tipo_trabajo:")
display(df_ot["tipo_trabajo"].value_counts(dropna=False).to_frame("ordenes"))

,tipo_python,nulos,valores_unicos
num_ot,int64,0,4427
objeto,string,0,166
desc_tipo_objeto,string,858,19
total_hh,float64,0,44
tipo_trabajo,string,0,20


Distribución de tipo_trabajo:


,ordenes
tipo_trabajo,
MP,3057
SER,409
MC,247
INS,216
FA,174
LIM,127
ALM_CORRECT,59
REP,42
R,28


## 3. Creación de la tabla en DuckDB

DuckDB ejecutará SQL directamente dentro de Google Colab. La conexión se crea en memoria y no necesita servidor, usuario ni contraseña.

La tabla `ordenes_trabajo` se reconstruye desde el dataframe preparado cada vez que ejecutas estas celdas.

In [8]:
# Crea una conexión temporal de DuckDB en memoria.
con = duckdb.connect(database=":memory:")

print("Conexión DuckDB creada correctamente.")

Conexión DuckDB creada correctamente.


In [9]:
# Registra temporalmente el dataframe para que DuckDB pueda leerlo.
con.register("df_ot", df_ot)

con.execute("""
    CREATE OR REPLACE TABLE ordenes_trabajo AS
    SELECT *
    FROM df_ot
""")

con.unregister("df_ot")

print("Tabla creada: ordenes_trabajo")

Tabla creada: ordenes_trabajo


In [10]:
consulta_control = con.execute("""
    SELECT
        COUNT(*) AS filas_duckdb,
        COUNT(DISTINCT num_ot) AS ots_unicas,
        COUNT(DISTINCT desc_tipo_objeto) AS tipos_equipo
    FROM ordenes_trabajo
""").fetchdf()

display(consulta_control)

assert int(consulta_control.loc[0, "filas_duckdb"]) == len(df_ot)
assert int(consulta_control.loc[0, "ots_unicas"]) == df_ot["num_ot"].nunique()

print("Validación de carga superada.")

,filas_duckdb,ots_unicas,tipos_equipo
0,4427,4427,20


Validación de carga superada.


# 4. Ejercicio integrador final

Construye **una consulta SQL para DuckDB** que genere un reporte agrupado por `desc_tipo_objeto`.

El resultado debe contener, exactamente en este orden:

1. `desc_tipo_objeto`
2. `total_ots_ejecutadas`
3. `num_fallas_correctivas`
4. `num_preventivos`
5. `total_horas_hombre`
6. `mttr_aprox_hh`
7. `ratio_correctivo_preventivo`
8. `clasificacion_criticidad`
9. `ranking_por_fallas`

## Definición de indicadores

- `total_ots_ejecutadas`: cantidad de órdenes del tipo de equipo.
- `num_fallas_correctivas`: cantidad de órdenes con `tipo_trabajo = 'MC'`.
- `num_preventivos`: cantidad de órdenes con `tipo_trabajo = 'MP'`.
- `total_horas_hombre`: suma de `total_hh`.
- `mttr_aprox_hh`: promedio de `total_hh` exclusivamente para órdenes `MC`; debe quedar `NULL` cuando el grupo no tenga correctivos.
- `ratio_correctivo_preventivo`: correctivos divididos entre preventivos. Usa `NULLIF()` para proteger la división entre cero.

## Criticidad

- `CRITICO`: cinco o más fallas correctivas.
- `IMPORTANTE`: entre dos y cuatro fallas correctivas.
- `NORMAL`: menos de dos fallas correctivas.

## Reglas obligatorias

- Excluir filas donde `total_hh`, `objeto` o `desc_tipo_objeto` sean `NULL`.
- Conservar solamente grupos con al menos diez órdenes mediante `HAVING`.
- Calcular `ranking_por_fallas` con `RANK()` sobre el número de correctivos, en orden descendente.
- Ordenar por `ranking_por_fallas` ascendente y después por `total_horas_hombre` descendente.
- No escribir manualmente nombres de tipos de equipo ni resultados numéricos.

> **Sugerencia:** una CTE puede separar la agregación de la clasificación y del ranking. La sugerencia no es obligatoria.

## 5. Rúbrica de evaluación

| Criterio | Ponderación |
|---|---:|
| Correctitud de KPIs y métricas | 40% |
| Clasificación de criticidad | 20% |
| Ranking por fallas | 15% |
| Condiciones de filtrado | 10% |
| Funcionalidades SQL avanzadas | 10% |
| Formato y orden de salida | 5% |
| **Total** | **100%** |

El calificador compara el resultado con los datos fuente y revisa la presencia de las construcciones SQL obligatorias. La revisión del profesor prevalece ante intentos equivalentes que el análisis automático no reconozca.

## 6. Zona de respuesta

Escribe tu consulta entre las comillas triples. No cambies el nombre `CONSULTA_ESTUDIANTE`.

En DuckDB, la ejecución debe conservar el formato:

```python
resultado = con.execute(CONSULTA_ESTUDIANTE).fetchdf()
```

In [12]:
CONSULTA_ESTUDIANTE = """
WITH base_agregada AS (
    SELECT
        desc_tipo_objeto,
        COUNT(*) AS total_ots_ejecutadas,
        SUM(CASE WHEN tipo_trabajo = 'MC' THEN 1 ELSE 0 END) AS num_fallas_correctivas,
        SUM(CASE WHEN tipo_trabajo = 'MP' THEN 1 ELSE 0 END) AS num_preventivos,
        SUM(total_hh) AS total_horas_hombre,
        AVG(CASE WHEN tipo_trabajo = 'MC' THEN total_hh ELSE NULL END) AS mttr_aprox_hh
    FROM
        ordenes_trabajo
    WHERE
        total_hh IS NOT NULL
        AND objeto IS NOT NULL
        AND desc_tipo_objeto IS NOT NULL
    GROUP BY
        desc_tipo_objeto
    HAVING
        COUNT(*) >= 10
)
SELECT
    desc_tipo_objeto,
    total_ots_ejecutadas,
    num_fallas_correctivas,
    num_preventivos,
    total_horas_hombre,
    mttr_aprox_hh,
    CAST(num_fallas_correctivas AS DOUBLE) / NULLIF(CAST(num_preventivos AS DOUBLE), 0) AS ratio_correctivo_preventivo,
    CASE
        WHEN num_fallas_correctivas >= 5 THEN 'CRITICO'
        WHEN num_fallas_correctivas >= 2 THEN 'IMPORTANTE'
        ELSE 'NORMAL'
    END AS clasificacion_criticidad,
    RANK() OVER (ORDER BY num_fallas_correctivas DESC) AS ranking_por_fallas
FROM
    base_agregada
ORDER BY
    ranking_por_fallas ASC,
    total_horas_hombre DESC
;"""

In [13]:
if not CONSULTA_ESTUDIANTE.strip():
    print("Escribe tu consulta SQL en la celda anterior y vuelve a ejecutar.")
else:
    resultado = con.execute(CONSULTA_ESTUDIANTE).fetchdf()
    print(f"Filas obtenidas: {len(resultado)}")
    display(resultado)

Filas obtenidas: 18


,desc_tipo_objeto,total_ots_ejecutadas,num_fallas_correctivas,num_preventivos,total_horas_hombre,mttr_aprox_hh,ratio_correctivo_preventivo,clasificacion_criticidad,ranking_por_fallas
0,<NA>,858,111.0000,109.0000,"2,432.2000",1.7477,1.0183,CRITICO,1
1,Bomba,1686,32.0000,"1,485.0000","7,571.5000",1.0000,0.0215,CRITICO,2
2,Estructural,356,26.0000,125.0000,"2,052.5000",0.1538,0.2080,CRITICO,3
3,Transformador,112,20.0000,71.0000,583.0000,0.9000,0.2817,CRITICO,4
4,Electrico,311,19.0000,273.0000,"1,396.0000",0.6842,0.0696,CRITICO,5
5,Perforadoras,44,15.0000,2.0000,89.4000,0.2933,7.5000,CRITICO,6
6,Compresores,103,8.0000,81.0000,216.0000,0.0000,0.0988,CRITICO,7
7,Sopladores y ventiladores g,190,6.0000,174.0000,650.0000,1.0000,0.0345,CRITICO,8
8,Distribución de energía eléctrica...,55,2.0000,47.0000,367.0000,0.0000,0.0426,IMPORTANTE,9
9,EXTRACTOR DE AIRE,71,2.0000,67.0000,137.0000,0.0000,0.0299,IMPORTANTE,9


## 7. Calificador automático

Ejecuta las dos celdas siguientes después de obtener `resultado`. El calificador no revela la consulta de referencia; verifica las métricas directamente contra el dataframe fuente.

In [14]:
def _serie_numerica(tabla, columna):
    return pd.to_numeric(tabla[columna], errors="coerce").astype(float)


def _coinciden_numeros(obtenido, esperado, columna, tolerancia=1e-6):
    izquierda = _serie_numerica(obtenido, columna)
    derecha = _serie_numerica(esperado, columna)
    return bool(
        np.allclose(
            izquierda.to_numpy(),
            derecha.to_numpy(),
            rtol=tolerancia,
            atol=tolerancia,
            equal_nan=True,
        )
    )


def _referencia_desde_fuente(datos):
    base = datos.loc[
        datos["total_hh"].notna()
        & datos["objeto"].notna()
        & datos["desc_tipo_objeto"].notna()
    ].copy()

    resumen = (
        base.groupby("desc_tipo_objeto", as_index=False)
        .agg(
            total_ots_ejecutadas=("num_ot", "size"),
            num_fallas_correctivas=(
                "tipo_trabajo",
                lambda serie: int(serie.eq("MC").sum()),
            ),
            num_preventivos=(
                "tipo_trabajo",
                lambda serie: int(serie.eq("MP").sum()),
            ),
            total_horas_hombre=("total_hh", "sum"),
        )
    )

    mttr = (
        base.loc[base["tipo_trabajo"].eq("MC")]
        .groupby("desc_tipo_objeto", as_index=False)["total_hh"]
        .mean()
        .rename(columns={"total_hh": "mttr_aprox_hh"})
    )

    resumen = resumen.merge(mttr, on="desc_tipo_objeto", how="left")
    resumen = resumen.loc[resumen["total_ots_ejecutadas"] >= 10].copy()

    denominador = resumen["num_preventivos"].replace(0, np.nan)
    resumen["ratio_correctivo_preventivo"] = (
        resumen["num_fallas_correctivas"] / denominador
    )

    resumen["clasificacion_criticidad"] = np.select(
        [
            resumen["num_fallas_correctivas"] >= 5,
            resumen["num_fallas_correctivas"] >= 2,
        ],
        ["CRITICO", "IMPORTANTE"],
        default="NORMAL",
    )

    resumen["ranking_por_fallas"] = (
        resumen["num_fallas_correctivas"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    columnas = [
        "desc_tipo_objeto",
        "total_ots_ejecutadas",
        "num_fallas_correctivas",
        "num_preventivos",
        "total_horas_hombre",
        "mttr_aprox_hh",
        "ratio_correctivo_preventivo",
        "clasificacion_criticidad",
        "ranking_por_fallas",
    ]

    return (
        resumen[columnas]
        .sort_values(
            ["ranking_por_fallas", "total_horas_hombre"],
            ascending=[True, False],
            kind="stable",
        )
        .reset_index(drop=True)
    )


def calificar_resultado(consulta, resultado_sql, datos_fuente):
    columnas_esperadas = [
        "desc_tipo_objeto",
        "total_ots_ejecutadas",
        "num_fallas_correctivas",
        "num_preventivos",
        "total_horas_hombre",
        "mttr_aprox_hh",
        "ratio_correctivo_preventivo",
        "clasificacion_criticidad",
        "ranking_por_fallas",
    ]

    referencia = _referencia_desde_fuente(datos_fuente)
    obtenido = resultado_sql.copy()
    obtenido.columns = [str(columna).lower() for columna in obtenido.columns]

    filas_rubrica = []

    def registrar(criterio, puntos, maximo, detalle):
        filas_rubrica.append(
            {
                "criterio": criterio,
                "puntos": round(float(puntos), 2),
                "máximo": float(maximo),
                "detalle": detalle,
            }
        )

    estructura_completa = all(
        columna in obtenido.columns for columna in columnas_esperadas
    )

    if estructura_completa:
        obtenido_comparacion = obtenido[columnas_esperadas].copy()
        obtenido_comparacion = obtenido_comparacion.sort_values(
            "desc_tipo_objeto", kind="stable"
        ).reset_index(drop=True)
        referencia_comparacion = referencia.sort_values(
            "desc_tipo_objeto", kind="stable"
        ).reset_index(drop=True)

        mismos_grupos = (
            len(obtenido_comparacion) == len(referencia_comparacion)
            and obtenido_comparacion["desc_tipo_objeto"].astype(str).tolist()
            == referencia_comparacion["desc_tipo_objeto"].astype(str).tolist()
        )
    else:
        obtenido_comparacion = pd.DataFrame()
        referencia_comparacion = referencia
        mismos_grupos = False

    pesos_kpi = {
        "total_ots_ejecutadas": 7,
        "num_fallas_correctivas": 7,
        "num_preventivos": 6,
        "total_horas_hombre": 7,
        "mttr_aprox_hh": 7,
        "ratio_correctivo_preventivo": 6,
    }

    puntos_kpi = 0
    aciertos_kpi = []
    for columna, peso in pesos_kpi.items():
        correcto = (
            mismos_grupos
            and _coinciden_numeros(
                obtenido_comparacion,
                referencia_comparacion,
                columna,
            )
        )
        if correcto:
            puntos_kpi += peso
            aciertos_kpi.append(columna)

    registrar(
        "1. Correctitud de KPIs y métricas",
        puntos_kpi,
        40,
        f"KPIs correctos: {len(aciertos_kpi)}/6",
    )

    criticidad_correcta = (
        mismos_grupos
        and obtenido_comparacion["clasificacion_criticidad"].astype(str).tolist()
        == referencia_comparacion["clasificacion_criticidad"].astype(str).tolist()
    )
    registrar(
        "2. Clasificación de criticidad",
        20 if criticidad_correcta else 0,
        20,
        "Correcta" if criticidad_correcta else "Revisar límites y etiquetas",
    )

    ranking_correcto = (
        mismos_grupos
        and _coinciden_numeros(
            obtenido_comparacion,
            referencia_comparacion,
            "ranking_por_fallas",
        )
    )
    registrar(
        "3. Ranking por fallas",
        15 if ranking_correcto else 0,
        15,
        "Correcto" if ranking_correcto else "Revisar RANK() y empates",
    )

    sql = " ".join(str(consulta).upper().split())
    comprobaciones_filtro = {
        "total_hh IS NOT NULL": "TOTAL_HH IS NOT NULL" in sql,
        "objeto IS NOT NULL": "OBJETO IS NOT NULL" in sql,
        "desc_tipo_objeto IS NOT NULL": "DESC_TIPO_OBJETO IS NOT NULL" in sql,
        "HAVING COUNT >= 10": (
            "HAVING" in sql and "COUNT" in sql and ">= 10" in sql
        ),
    }
    puntos_filtro = 2.5 * sum(comprobaciones_filtro.values())
    registrar(
        "4. Condiciones de filtrado",
        puntos_filtro,
        10,
        "; ".join(
            f"{nombre}: {'OK' if valor else 'FALTA'}"
            for nombre, valor in comprobaciones_filtro.items()
        ),
    )

    comprobaciones_sql = {
        "GROUP BY": "GROUP BY" in sql,
        "HAVING": "HAVING" in sql,
        "CASE": "CASE" in sql,
        "RANK() OVER": "RANK()" in sql and "OVER" in sql,
        "NULLIF": "NULLIF" in sql,
    }
    puntos_sql = 2 * sum(comprobaciones_sql.values())
    registrar(
        "5. Funcionalidades SQL avanzadas",
        puntos_sql,
        10,
        "; ".join(
            f"{nombre}: {'OK' if valor else 'FALTA'}"
            for nombre, valor in comprobaciones_sql.items()
        ),
    )

    columnas_en_orden = obtenido.columns.tolist() == columnas_esperadas
    if estructura_completa:
        orden_obtenido = obtenido[columnas_esperadas].reset_index(drop=True)
        orden_esperado = obtenido[columnas_esperadas].sort_values(
            ["ranking_por_fallas", "total_horas_hombre"],
            ascending=[True, False],
            kind="stable",
        ).reset_index(drop=True)
        salida_ordenada = orden_obtenido.equals(orden_esperado)
    else:
        salida_ordenada = False

    puntos_formato = (3 if columnas_en_orden else 0) + (2 if salida_ordenada else 0)
    registrar(
        "6. Formato y orden de salida",
        puntos_formato,
        5,
        f"Columnas: {'OK' if columnas_en_orden else 'REVISAR'}; "
        f"orden: {'OK' if salida_ordenada else 'REVISAR'}",
    )

    rubrica = pd.DataFrame(filas_rubrica)
    total = rubrica["puntos"].sum()
    rubrica.loc[len(rubrica)] = {
        "criterio": "CALIFICACIÓN TOTAL",
        "puntos": total,
        "máximo": 100,
        "detalle": "Evaluación automática; sujeta a revisión del profesor",
    }

    return rubrica


In [15]:
if "resultado" not in globals():
    print("Primero ejecuta tu consulta para crear el dataframe 'resultado'.")
else:
    calificacion = calificar_resultado(
        CONSULTA_ESTUDIANTE,
        resultado,
        df_ot,
    )
    display(calificacion)
    print(
        f"Calificación automática: "
        f"{calificacion.iloc[-1]['puntos']:.2f} / 100"
    )

,criterio,puntos,máximo,detalle
0,1. Correctitud de KPIs y métricas,0.0000,40.0000,KPIs correctos: 0/6
1,2. Clasificación de criticidad,0.0000,20.0000,Revisar límites y etiquetas
2,3. Ranking por fallas,0.0000,15.0000,Revisar RANK() y empates
3,4. Condiciones de filtrado,10.0000,10.0000,total_hh IS NOT NULL: OK; objeto IS NOT NULL: ...
4,5. Funcionalidades SQL avanzadas,10.0000,10.0000,GROUP BY: OK; HAVING: OK; CASE: OK; RANK() OVE...
5,6. Formato y orden de salida,5.0000,5.0000,Columnas: OK; orden: OK
6,CALIFICACIÓN TOTAL,25.0000,100.0000,Evaluación automática; sujeta a revisión del p...


Calificación automática: 25.00 / 100


## 8. Lista de comprobación antes de entregar

- [ ] El CSV fue subido correctamente a Google Colab.
- [ ] DuckDB contiene 4,427 filas y 4,427 OTs únicas.
- [ ] La consulta se ejecuta completa sin errores.
- [ ] El resultado contiene las nueve columnas en el orden solicitado.
- [ ] `mttr_aprox_hh` considera solamente órdenes `MC`.
- [ ] El ratio está protegido con `NULLIF()`.
- [ ] La criticidad respeta exactamente los tres intervalos.
- [ ] El ranking fue construido con `RANK()` y conserva los empates.
- [ ] Los tres filtros `IS NOT NULL` están escritos en SQL.
- [ ] El filtro de grupos utiliza `HAVING COUNT(*) >= 10`.
- [ ] El resultado final tiene el orden solicitado.
- [ ] El notebook conserva la consulta, el resultado y la calificación automática.

Selecciona **Archivo > Descargar > Descargar .ipynb** antes de cerrar Google Colab.